In [ ]:
import pandas as pd
from sqlalchemy import create_engine
# Create a connection to the PostgreSQL database
engine = create_engine('postgresql+psycopg2://postgres:Zxcvbnm,./123@localhost:5432/procurement_db')

# Import csv File
df = pd.read_csv(r"C:\Users\HP\Desktop\Procurement_Raw_Data.csv")

df.to_sql("supplier_performance", engine, if_exists="replace", index=False)
print("File Loaded Successfully")
print(df.shape)
print(df.columns.to_list())

File Loaded Successfully
(777, 11)
['PO_ID', 'Supplier', 'Order_Date', 'Delivery_Date', 'Item_Category', 'Order_Status', 'Quantity', 'Unit_Price', 'Negotiated_Price', 'Defective_Units', 'Compliance']


In [2]:
# Correct The table
from sqlalchemy import text
query = """

ALTER TABLE supplier_performance RENAME COLUMN "PO_ID" TO Po_id;
ALTER TABLE supplier_performance RENAME COLUMN "Supplier" TO Supplier;
ALTER TABLE supplier_performance RENAME COLUMN "Order_Date" TO Order_Date;
ALTER TABLE supplier_performance RENAME COLUMN "Delivery_Date" TO Delivery_Date;
ALTER TABLE supplier_performance RENAME COLUMN "Item_Category" TO Item_category;
ALTER TABLE supplier_performance RENAME COLUMN "Order_Status" TO Order_Status;
ALTER TABLE supplier_performance RENAME COLUMN "Quantity" TO Quantity;
ALTER TABLE supplier_performance RENAME COLUMN "Unit_Price" TO Unit_Price;
ALTER TABLE supplier_performance RENAME COLUMN "Negotiated_Price" TO Negotiated_Price;
ALTER TABLE supplier_performance RENAME COLUMN "Defective_Units" TO Defective_Units;
ALTER TABLE supplier_performance RENAME COLUMN "Compliance" TO Compliance;
"""

# Execute SQL commands
with engine.begin() as connection:
    for statement in query.strip().split(";"):
        if statement.strip():
            connection.execute(text(statement))

print("Columns renamed successfully!")


Columns renamed successfully!


In [3]:
# Overall Summary
query1 = """
SELECT COUNT(*) AS Total_Orders, COUNT(DISTINCT supplier) as Total_Suppliers,
SUM (Quantity) AS Total_quantity,
SUM (defective_units) AS Total_defective_units
FROM supplier_Performance;
"""
Overall = pd.read_sql(query1, engine)
print(Overall)

   total_orders  total_suppliers  total_quantity  total_defective_units
0           777                5        850551.0                47949.0


In [4]:
# Task 2 Order Status Breakdown
query2 = """
SELECT 
    Order_Status,
    COUNT(*) AS total_orders,
    SUM(Quantity) AS total_units,
    SUM(Defective_Units) AS total_defective_units,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM supplier_performance
GROUP BY Order_Status
ORDER BY total_orders DESC;
"""
Status_analysis = pd.read_sql(query2, engine)
print(Status_analysis)

          order_status  total_orders  total_units  total_defective_units  \
0            Delivered           560     614078.0                33463.0   
1              Pending            81      92420.0                 5155.0   
2  Partially Delivered            73      78617.0                 5050.0   
3            Cancelled            63      65436.0                 4281.0   

   percentage  
0       72.07  
1       10.42  
2        9.40  
3        8.11  


In [5]:
# Task 2 Supplier Performance (efect Rate by Supplier
query3 = """
SELECT Supplier,
    COUNT(*) AS total_orders,
    ROUND((SUM(Defective_Units) * 100.0 / SUM(Quantity))::numeric, 2) AS defect_rate
FROM supplier_performance
GROUP BY Supplier
ORDER BY defect_rate ASC;
"""
Supplier_performance = pd.read_sql(query3, engine)
print(Supplier_performance)

          supplier  total_orders  defect_rate
0        Alpha_Inc           141         1.81
1    Epsilon_Group           166         2.60
2         Gamma_Co           143         4.45
3    Beta_Supplies           156         7.67
4  Delta_Logistics           171        10.83


In [6]:
# Task 2 Supplier Performance (Overall Defect Rate)
query4 = """
SELECT 
    COUNT(*) AS total_orders,
    SUM(Quantity) AS total_quantity,
    SUM(Defective_Units) AS total_defective_units,
    ROUND((SUM(Defective_Units) * 100.0 / SUM(Quantity))::numeric, 2) AS defect_rate
FROM supplier_performance;
"""
Overall_defect_rate = pd.read_sql(query4, engine)
print(Overall_defect_rate)

   total_orders  total_quantity  total_defective_units  defect_rate
0           777        850551.0                47949.0         5.64


In [7]:
# Task 3 Compliance Analysis (by supplier)
query5 = """
SELECT      
    Supplier,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN Compliance = 'Yes' THEN 1 ELSE 0 END) AS compliant_orders,
    ROUND(SUM(CASE WHEN Compliance = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS compliance_rate
FROM supplier_performance
GROUP BY Supplier
ORDER BY compliance_rate DESC;
"""
Compliance_supplier = pd.read_sql(query5, engine)
print(Compliance_supplier)

          supplier  total_orders  compliant_orders  compliance_rate
0    Epsilon_Group           166               163            98.19
1        Alpha_Inc           141               132            93.62
2         Gamma_Co           143               123            86.01
3    Beta_Supplies           156               118            75.64
4  Delta_Logistics           171               104            60.82


In [8]:
# Compliance Analysis (Overall Compliance)
query6 = """
SELECT compliance, COUNT(*) AS Total_orders,
ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM supplier_performance 
GROUP BY compliance
ORDER BY Total_Orders DESC;
"""
Overall_comp = pd.read_sql(query6, engine)
print(Overall_comp)

  compliance  total_orders  percentage
0        Yes           640       82.37
1         No           137       17.63


In [9]:
# Task 4 Delivery Delay Aanalysis (Average Delivery Days by Supplier)
query7 = """
SELECT 
    Supplier,
    COUNT(*) AS total_orders,
    ROUND(AVG(Delivery_Date::date - Order_Date::date)::numeric, 2) AS avg_delivery_days
FROM supplier_performance
GROUP BY Supplier
ORDER BY avg_delivery_days;
"""
Avg_days = pd.read_sql(query7, engine)
print(Avg_days)

          supplier  total_orders  avg_delivery_days
0         Gamma_Co           143              10.19
1        Alpha_Inc           141              10.61
2  Delta_Logistics           171              10.85
3    Epsilon_Group           166              10.87
4    Beta_Supplies           156              11.27


In [10]:
# Task 4 Delivery Delay Analysis (Overall Delay)
query8 = """
SELECT ROUND(AVG(Delivery_Date::date - Order_Date::date)::numeric, 2) AS avg_delivery_days
FROM supplier_performance;
"""
Overall_delay = pd.read_sql(query8, engine)
print(Overall_delay)

   avg_delivery_days
0              10.78


In [11]:
# Task 5 Incomplete Orders
query9 = """
    SELECT Supplier,
    COUNT(*) AS incomplete_orders
FROM supplier_performance
WHERE Order_Status IN ('Partially Delivered', 'Pending', 'Cancelled')
GROUP BY Supplier
ORDER BY incomplete_orders DESC;
"""
Incomplete_Orders = pd.read_sql(query9, engine)
print(Incomplete_Orders)


          supplier  incomplete_orders
0  Delta_Logistics                 51
1    Beta_Supplies                 46
2    Epsilon_Group                 46
3         Gamma_Co                 40
4        Alpha_Inc                 34
